# USV Summary Statistics — Plots

This notebook produces every figure that summarizes a single recording session's ultrasonic vocalizations (USVs). It runs **after** the processing pipeline has produced (a) a `*_usv_summary.csv` (per-USV start/stop/duration/category/emitter) and (b) the 3D translated/rotated/metric tracks for each animal.

The cells are organised as: *(1) load + configure*, *(2) per-figure compute + plot*. Each plotting cell is independent — you can re-run any single one without re-running the rest, provided cells **[1] Extract data** and **[2] Define figure save directory** have run.


In [ ]:
### Imports and color settings

from __future__ import annotations

import json
from pathlib import Path
import re

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pls

from usv_playpen.os_utils import configure_path
from usv_playpen.visualizations.auxiliary_plot_functions import create_colormap
import usv_playpen.visualizations.usv_summary_statistics as uss

base_path = Path.cwd().parent
fm.fontManager.addfont(base_path / "fonts/Helvetica.ttf")
plt.style.use(base_path / "_config/usv_playpen.mplstyle")

with open(Path.cwd().parent / "_parameter_settings" / "visualizations_settings.json", 'r') as vis_settings_file:
    vis_settings = json.load(vis_settings_file)

male_color = vis_settings["male_colors"][0]
female_color = vis_settings["female_colors"][0]
unassigned_color = "#C0C0C0"
line_color = "#202020"

female_cmap = create_colormap(input_parameter_dict={
    "cm_length": 255,
    "cm_name": "female_cm",
    "cm_type": "sequential",
    "cm_start": (
        int(female_color[1:3], 16),
        int(female_color[3:5], 16),
        int(female_color[5:7], 16),
    ),
    "cm_end": (255, 255, 255),
    "equalize_luminance": True,
    "match_luminance_by": "max",
    "change_saturation": 0.5,
    "cm_opacity": 1,
})

male_cmap = create_colormap(input_parameter_dict={
    "cm_length": 255,
    "cm_name": "male_cm", 
    "cm_type": "sequential",
    "cm_start": (
        int(male_color[1:3], 16),
        int(male_color[3:5], 16),
        int(male_color[5:7], 16),
    ),
    "cm_end": (255, 255, 255),
    "equalize_luminance": True,
    "match_luminance_by": "max",
    "change_saturation": 0.5,
    "cm_opacity": 1,
})

## 1. Load the session

Read the `_usv_summary.csv` and the 3D track HDF5 into memory and decode session metadata (cohort, experiment code, estrous status, …) from the directory name. Every downstream cell consumes these in-memory objects.


In [ ]:
### Extract data

txt_sessions_file = Path(configure_path('/mnt/falkner/Bartul/modeling/input_files/courtship_behavioral_intact_partners_sessions_list.txt'))
session_type = re.sub(r"courtship_behavioral_|_list.txt", "", txt_sessions_file.name)
noise_col_id = 'usv_supercategory'
noise_categories = [0]
usv_category_col = 'usv_supercategory'
distance_suffix = 'nose-nose'
mf_angle_suffix = 'allo_yaw-nose'
fm_angle_suffix = 'nose-allo_yaw'

with txt_sessions_file.open('r') as sessions_txt_file:
    session_roots = [line.strip() for line in sessions_txt_file if line.strip()]

print(f"Found {len(session_roots)} sessions in the text file.")

usv_pls, bg_pls, noise_filtered_count = uss.build_master_usv_dataframe(
    session_roots=session_roots,
    noise_col_id=noise_col_id,
    noise_categories=noise_categories,
    usv_category_col=usv_category_col,
    distance_suffix=distance_suffix,
    mf_angle_suffix=mf_angle_suffix,
    fm_angle_suffix=fm_angle_suffix
)

usv_df = usv_pls.to_pandas()
usv_df['duration_ms'] = usv_df['duration'] * 1000

valid_vocalizations_count = len(usv_df)
noise_proportion = noise_filtered_count / valid_vocalizations_count if valid_vocalizations_count > 0 else 0.0

print("Extraction complete!")
print(f"Total valid vocalizations loaded (M+F+U): {valid_vocalizations_count}")
print(f"Total noise vocalizations filtered out: {noise_filtered_count}")
print(f"Noise proportion in total data: {noise_proportion:.2%}")
display(usv_df.head())


## 2. Output directory

All figures from this notebook are written to a single per-session directory; the path is logged so you can find the file on disk after running.


In [ ]:
### Define figure save directory

figure_save_dir = Path(configure_path('/mnt/falkner/Bartul/figures'))
figure_save_dir.mkdir(parents=True, exist_ok=True)

save_fig_bool = False
save_fig_format = 'svg'

## 3. Per-session assignment summary

How many vocalizations were attributed to mouse 1, mouse 2, both, or neither? This is the first sanity check after running the assignment step — a session with ≫50% unassigned USVs usually indicates a sound-localization problem upstream.


In [ ]:
### Plot assignment statistics

assignment_df = (
    usv_pls
    .group_by(['session_id', 'sex'])
    .agg(pls.len().alias('count'))
    .pivot(values='count', index='session_id', on='sex')
    .fill_null(0)
)
for col in ['male', 'female', 'unassigned']:
    if col not in assignment_df.columns:
        assignment_df = assignment_df.with_columns(pls.lit(0).alias(col))
assignment_df = assignment_df.rename({'session_id': 'session'})

# Plot raw counts
fig_bars_raw, ax_bars_raw, stats_bars_raw = uss.plot_assignment_stacked_bars(
    assignment_df=assignment_df,
    plot_proportions=False,
    male_color=male_color,
    female_color=female_color,
    unassigned_color=unassigned_color
)
ax_bars_raw.set_title("Assignment Breakdown Raw")

if save_fig_bool:
    fig_bars_raw.savefig(figure_save_dir / f"{session_type}_usv_assignment_raw_counts.{save_fig_format}", bbox_inches='tight')
plt.show()

# Plot proportions
fig_bars_prop, ax_bars_prop, stats_bars_prop = uss.plot_assignment_stacked_bars(
    assignment_df=assignment_df,
    plot_proportions=True,
    male_color=male_color,
    female_color=female_color,
    unassigned_color=unassigned_color
)
ax_bars_prop.set_title("Assignment Breakdown Proportions")

if save_fig_bool:
    fig_bars_prop.savefig(figure_save_dir / f"{session_type}_usv_assignment_proportions.{save_fig_format}", bbox_inches='tight')
plt.show()

print("Stacked bar stats:")
for key, value in stats_bars_raw.items():
    print(f"{key}: {value}")

fig_summary, axes_summary, stats_summary = uss.plot_assignment_summary_panel(
    assignment_df=assignment_df,
    male_color=male_color,
    female_color=female_color,
    unassigned_color=unassigned_color,
    jitter_strength=0.1
)

if save_fig_bool:
    fig_summary.savefig(figure_save_dir / f"{session_type}_usv_global_assignment_summary.{save_fig_format}", bbox_inches='tight')
plt.show()

print("Global assignment statistics (noise filtered):")
print(f"Grand total non-noise USVs: {stats_summary['grand_total']:.0f}")
print(f"Male total: {stats_summary['male_total']:.0f} ({stats_summary['male_proportion']:.2%})")
print(f"Female total: {stats_summary['female_total']:.0f} ({stats_summary['female_proportion']:.2%})")
print(f"Unassigned total: {stats_summary['unassigned_total']:.0f} ({stats_summary['unassigned_proportion']:.2%})")


## 4. Assignment status by USV category

Conditioned on the DAS-predicted USV category (call type), what fraction is assignable? If certain categories are systematically harder to localize (e.g. very short calls, low-SNR calls), they will appear here as outliers in the unassigned bar.


In [ ]:
### Plot relationship between USV categories and assignment status

usv_category_col = 'usv_supercategory'
usv_continuous_cols = ('usv_umap1', 'usv_umap2')

print("Extracting embedding data...")
df_embedding = uss.extract_category_embedding_data(
    session_roots=session_roots,
    noise_col_id=noise_col_id,
    noise_categories=noise_categories,
    usv_category_col=usv_category_col,
    usv_continuous_cols=usv_continuous_cols
)

if not df_embedding.is_empty():
    fig_emb, ax_emb = uss.plot_category_prevalence_and_embedding(
        df_embedding=df_embedding,
        male_color=male_color,
        female_color=female_color,
        unassigned_color=unassigned_color,
        boundary_color='#000000',
        log_scale_bars=False,
        plot_type='density',
        grid_res=400
    )
    
    if save_fig_bool:
        fig_emb.savefig(figure_save_dir / f"{session_type}_usv_category_embedding_assignment_panel.{save_fig_format}", bbox_inches='tight', dpi=300)
    plt.show()
else:
    print("No valid embedding data found. Check your column names.")

## 5. Per-mouse participation

For each tracked animal, what share of the assigned USVs did it emit? Useful for courtship sessions where one expects a strong sex-asymmetry in emission rate.


In [ ]:
### Plot participation of individual mice

print(f"Processing {len(session_roots)} sessions for animal statistics...")

# Count sessions per animal ID (one entry per session regardless of USV count)
male_sessions = (
    usv_pls
    .select(['session_id', 'male_id'])
    .unique()
    .group_by('male_id')
    .agg(pls.len().alias('session_count'))
)
male_usv_counts = (
    usv_pls
    .filter(pls.col('sex') == 'male')
    .group_by('male_id')
    .agg(pls.len().alias('total_usvs'))
)
male_stats_pls = male_sessions.join(male_usv_counts, on='male_id', how='left').fill_null(0)
male_animal_stats = {
    row['male_id']: {'session_count': row['session_count'], 'total_usvs': row['total_usvs']}
    for row in male_stats_pls.iter_rows(named=True)
}

female_sessions = (
    usv_pls
    .select(['session_id', 'female_id'])
    .unique()
    .group_by('female_id')
    .agg(pls.len().alias('session_count'))
)
female_usv_counts = (
    usv_pls
    .filter(pls.col('sex') == 'female')
    .group_by('female_id')
    .agg(pls.len().alias('total_usvs'))
)
female_stats_pls = female_sessions.join(female_usv_counts, on='female_id', how='left').fill_null(0)
female_animal_stats = {
    row['female_id']: {'session_count': row['session_count'], 'total_usvs': row['total_usvs']}
    for row in female_stats_pls.iter_rows(named=True)
}

fig_male, axes_male, stats_male = uss.plot_animal_participation_stats(
    animal_stats=male_animal_stats,
    sex_label='Male',
    bar_color=male_color,
    text_color='#202020'
)

if save_fig_bool:
    fig_male.savefig(figure_save_dir / f"{session_type}_male_participation_stats.{save_fig_format}", bbox_inches='tight')
plt.show()

fig_female, axes_female, stats_female = uss.plot_animal_participation_stats(
    animal_stats=female_animal_stats,
    sex_label='Female',
    bar_color=female_color,
    text_color='#202020'
)

if save_fig_bool:
    fig_female.savefig(figure_save_dir / f"{session_type}_female_participation_stats.{save_fig_format}", bbox_inches='tight')
plt.show()

print("Animal participation summary:")
print(f"Unique males: {stats_male['total_animals']} | Avg sessions/male: {stats_male['mean_session_count']:.1f}")
print(f"Unique females: {stats_female['total_animals']} | Avg sessions/female: {stats_female['mean_session_count']:.1f}")


## 6. Global vocalization rate over time

Binned USV count vs. session time. Captures the typical "warm-up + decay" fatigue envelope: a brief burst at the start, a stable middle, and a decay toward the end.


In [ ]:
### Plot global vocalization fatigue

# Plot duration histograms
print("Plotting global duration distributions...")
fig_hist, axes_hist, stats_hist = uss.plot_duration_histograms_by_sex(
    plot_data=usv_df,
    bin_width_ms=20.0,
    max_duration_ms=300.0,
    male_color=male_color,
    female_color=female_color
)

if save_fig_bool:
    fig_hist.savefig(figure_save_dir / f"{session_type}_global_duration_histograms.{save_fig_format}", bbox_inches='tight')
plt.show()

# Plot hourly regression for USV duration
print("Plotting hourly regression (USV duration)...")
fig_hr_dur, axes_hr_dur, stats_hr_dur = uss.plot_hourly_regressions(
    df_raw=usv_df,
    y_col='duration_ms',
    y_label='USV duration (ms)',
    male_color=male_color,
    female_color=female_color,
    line_color='#202020'
)

if save_fig_bool:
    fig_hr_dur.savefig(figure_save_dir / f"{session_type}_global_hourly_duration_regression.{save_fig_format}", bbox_inches='tight')
plt.show()

# Plot hourly regression for USV count
print("Plotting hourly regression (USV count)...")
# Aggregate the global dataframe to get the count of USVs per hour for each session/sex
hourly_counts_df = usv_df.groupby(['session_id', 'hour', 'sex']).size().reset_index(name='usv_count')

fig_hr_cnt, axes_hr_cnt, stats_hr_cnt = uss.plot_hourly_regressions(
    df_raw=hourly_counts_df,
    y_col='usv_count',
    y_label='USV count/session',
    male_color=male_color,
    female_color=female_color,
    line_color='#202020'
)

if save_fig_bool:
    fig_hr_cnt.savefig(figure_save_dir / f"{session_type}_global_hourly_count_regression.{save_fig_format}", bbox_inches='tight')
plt.show()

## 7. Global rate broken down by category

Same as the previous panel, but stacked by call type — useful for spotting categories that decay faster than others (often the short / harmonic types).


In [ ]:
### Plot global fatigue broken down by USV category

print("Generating global category fatigue heatmaps...")

# The function expects a Polars DataFrame for efficient processing
fig_fatigue_heat, axes_fatigue_heat, stats_fatigue_heat = uss.plot_category_global_fatigue_heatmap(
    global_usv_df=usv_pls,
    smoothing_sigma=1.0,
    colormap='magma'
)

if save_fig_bool:
    fatigue_heat_path = figure_save_dir / f"{session_type}_usv_category_global_fatigue_heatmap.{save_fig_format}"
    fig_fatigue_heat.savefig(fatigue_heat_path, bbox_inches='tight', dpi=300)

plt.show()

## 8. Local fatigue around emission events

Conditioned on each USV onset, what's the rate in a small window around it? Pulls out *bursting* structure that the global plot averages away.


In [ ]:
### Plot local vocalization fatigue

bin_width_seconds = 120  # 2-minute bins
max_time_seconds = 1200  # First 20 minutes of the session
n_bins = max_time_seconds // bin_width_seconds

print(f"Extracting local fatigue data in {bin_width_seconds}s bins...")

binned_df_full = (
    usv_pls
    .filter(pls.col('sex').is_in(['male', 'female']))
    .with_columns(
        (pls.col('start') // bin_width_seconds).cast(pls.Int32).alias('time_bin')
    )
    .filter(pls.col('time_bin') < n_bins)
    .group_by(['session_id', 'sex', 'time_bin'])
    .agg(pls.len().alias('usv_count'))
    .to_pandas()
)

# Pad missing bins with zeros so silent bins count as 0, not missing
unique_sessions = binned_df_full['session_id'].unique()
sexes = ['male', 'female']
bin_indices = np.arange(n_bins)

multi_idx = pd.MultiIndex.from_product(
    [unique_sessions, sexes, bin_indices],
    names=['session_id', 'sex', 'time_bin']
)
binned_padded = (
    binned_df_full.set_index(['session_id', 'sex', 'time_bin'])
    .reindex(multi_idx, fill_value=0)
    .reset_index()
)

binned_agg = binned_padded.groupby(['sex', 'time_bin'])['usv_count'].agg(['mean', 'sem']).reset_index()
binned_agg.rename(columns={'mean': 'mean_usv', 'sem': 'sem_usv'}, inplace=True)

print("Plotting Local Fatigue Trends...")
fig_local, axes_local, stats_local = uss.plot_local_fatigue_binned_trends(
    binned_df=binned_agg,
    y_mean_col='mean_usv',
    y_sem_col='sem_usv',
    y_label=f'Mean USVs / {bin_width_seconds//60}-min',
    bin_width_seconds=bin_width_seconds,
    n_bins=n_bins,
    male_color=male_color,
    female_color=female_color,
    use_log_scale=False
)

if save_fig_bool:
    fig_local.savefig(figure_save_dir / f"{session_type}_local_fatigue_binned_trends.{save_fig_format}", bbox_inches='tight')
plt.show()


## 9. Local fatigue by category

Category-resolved version of the previous panel — shows whether bursts are category-homogeneous or mixed.


In [ ]:
### Plot local fatigue broken down by USV category

binned_df_categorical = (
    usv_pls
    .filter(pls.col('sex').is_in(['male', 'female']))
    .with_columns(
        (pls.col('start') // bin_width_seconds).cast(pls.Int32).alias('time_bin')
    )
    .filter(pls.col('time_bin') < n_bins)
    .group_by(['session_id', 'sex', 'category', 'time_bin'])
    .agg(pls.len().alias('usv_count'))
    .to_pandas()
)

print(f"Generating categorical heatmap for {len(binned_df_categorical)} data points...")

fig_heat, axes_heat, stats_heat = uss.plot_category_local_fatigue_heatmap(
    binned_df=binned_df_categorical,
    bin_width_seconds=bin_width_seconds,
    n_bins=n_bins,
    smoothing_sigma=1,
    colormap='magma',
    facet_figsize=(12, 10)
)

if save_fig_bool:
    fig_heat.savefig(figure_save_dir / f"{session_type}_usv_category_local_fatigue_heatmap.{save_fig_format}", bbox_inches='tight', dpi=300)
plt.show()

print("Peak mean USV rates per category (for normalization reference):")
display(pd.DataFrame(stats_heat))


## 10. Unassigned-rate vs. inter-animal proximity (session-level)

When the two mice are close together, sound localization fails more often. This cell quantifies the session-level correlation between mean nose–nose distance and the fraction of unassigned USVs.


In [ ]:
### Relationship between unassigned USVs and nose-nose distance (at the session level)

print("Extracting session-level distance and assignment proportions...")

df_jointplot = (
    usv_pls
    .filter(pls.col('distance').is_not_null())
    .group_by('session_id')
    .agg([
        pls.len().alias('total'),
        (pls.col('sex') == 'unassigned').sum().alias('unassigned_count'),
        pls.col('distance').median().alias('median_distance')
    ])
    .with_columns(
        (pls.col('unassigned_count') / pls.col('total')).alias('unassigned_prop')
    )
    .select(['median_distance', 'unassigned_prop'])
    .to_pandas()
    .dropna()
)

print(f"Aggregated data for {len(df_jointplot)} valid sessions.")

if not df_jointplot.empty:
    fig_joint, stats_joint = uss.plot_unassigned_proportion_vs_distance_jointplot(
        df_combined=df_jointplot,
        scatter_color='#808080',
        line_color='#FF0000',
        hist_color='#A0A0A0'
    )
    if save_fig_bool:
        fig_joint.savefig(figure_save_dir / f"{session_type}_unassigned_proportion_vs_distance_session_level.{save_fig_format}", bbox_inches='tight')
    plt.show()
else:
    print("Not enough data to plot.")


## 11. Unassigned-rate vs. proximity (per-USV level)

Same question at the per-call level: nose-nose distance at the moment of each USV vs. whether that USV was assignable.


In [ ]:
### Relationship between unassigned USVs and nose-nose distance (at the individual USV level)

print("Extracting per-USV spatial behavior...")

usv_categories_list = sorted(usv_df['category'].dropna().unique().tolist())

df_anova = (
    usv_pls
    .select(['sex', 'distance'])
    .drop_nulls()
    .rename({'sex': 'category'})
    .to_pandas()
)

print(f"Extracted {len(df_anova)} valid vocalizations with spatial coordinates.")
print("Running statistical tests and plotting KDE distributions...")

fig_anova, ax_anova, stats_anova = uss.plot_distance_by_assignment_kde_anova(
    df_plot=df_anova,
    min_samples_anova=30,
    male_color=male_color,
    female_color=female_color,
    unassigned_color=unassigned_color
)

if save_fig_bool:
    fig_anova.savefig(figure_save_dir / f"{session_type}_distance_by_assignment_kde_anova.{save_fig_format}", bbox_inches='tight')
plt.show()


## 12. USV duration vs. spatial behavior

A grid of marginal regressions of USV duration on each spatial / postural feature (speed, orientation to partner, head pitch, etc.). Helpful for spotting which behavioral covariates are predictive of call duration.


In [ ]:
### Plot relationship between spatial behavior and USV duration (regression grid)

male_reg_df = (
    usv_pls
    .filter(pls.col('sex') == 'male')
    .select(['distance', 'mf_angle', 'duration'])
    .drop_nulls()
    .rename({'mf_angle': 'angle', 'duration': 'usv_duration'})
    .to_pandas()
)

female_reg_df = (
    usv_pls
    .filter(pls.col('sex') == 'female')
    .select(['distance', 'fm_angle', 'duration'])
    .drop_nulls()
    .rename({'fm_angle': 'angle', 'duration': 'usv_duration'})
    .to_pandas()
)

print(f"Plotting regressions for {len(male_reg_df)} Male USVs and {len(female_reg_df)} Female USVs...")

fig_reg, axes_reg, stats_reg = uss.plot_behavior_duration_regressions(
    male_df=male_reg_df,
    female_df=female_reg_df,
    male_color=male_color,
    female_color=female_color,
    line_color='#202020'
)

if save_fig_bool:
    fig_reg.savefig(figure_save_dir / f"{session_type}_behavior_duration_regressions.{save_fig_format}", bbox_inches='tight')
plt.show()


## 13. Estrous-stage USV metrics

Aggregate USV rate / duration / inter-call interval grouped by the female's estrous stage decoded from the experiment code (proestrus / estrus / matestrus / diestrus).


In [ ]:
### Plot estrous stage USV metrics

usv_categories_list = sorted(usv_df['category'].dropna().unique().tolist())

estrous_code_index = -1
valid_stages = ['p', 'e', 'm', 'd']

label_map = {
    'p': 'Proestrus',
    'e': 'Estrus',
    'm': 'Metestrus',
    'd': 'Diestrus'
}

category_order = ['p', 'e', 'm', 'd']
category_labels = [label_map[stage] for stage in category_order]
estrous_colors = ['#810000', '#ff1714', '#ff5555', '#ffaaaa']

# Attach estrous stage to master DataFrame
usv_pls = usv_pls.with_columns(
    pls.col('experiment_code').str.slice(estrous_code_index).alias('estrous_stage')
)
usv_df = usv_pls.to_pandas()

estrous_subset = usv_pls.filter(pls.col('estrous_stage').is_in(valid_stages))

# Session counts per stage
session_counts = dict.fromkeys(valid_stages, 0)
for row in (
    estrous_subset
    .select(['session_id', 'estrous_stage'])
    .unique()
    .group_by('estrous_stage')
    .agg(pls.len().alias('n'))
    .iter_rows(named=True)
):
    session_counts[row['estrous_stage']] = row['n']

# Male and female USV counts per stage
male_usv_counts = dict.fromkeys(valid_stages, 0)
female_usv_counts = dict.fromkeys(valid_stages, 0)
for stage in valid_stages:
    stage_df = estrous_subset.filter(pls.col('estrous_stage') == stage)
    male_usv_counts[stage] = stage_df.filter(pls.col('sex') == 'male').height
    female_usv_counts[stage] = stage_df.filter(pls.col('sex') == 'female').height

# Per-session M:F ratios
male_female_ratios = {stage: [] for stage in valid_stages}
for (session_id_val, stage), grp in estrous_subset.to_pandas().groupby(['session_id', 'estrous_stage']):
    m = max((grp['sex'] == 'male').sum(), 1)
    f = max((grp['sex'] == 'female').sum(), 1)
    male_female_ratios[stage].append(m / f)

# Build estrous_data dict required by plot_category_estrous_rates_grid / ratio_grid
estrous_data = {}
for cat in usv_categories_list:
    cat_df = estrous_subset.filter(pls.col('category') == cat)
    cat_session_counts = dict.fromkeys(valid_stages, 0)
    cat_male_counts = dict.fromkeys(valid_stages, 0)
    cat_female_counts = dict.fromkeys(valid_stages, 0)
    cat_ratios = {stage: [] for stage in valid_stages}
    for stage in valid_stages:
        s_df = cat_df.filter(pls.col('estrous_stage') == stage)
        cat_session_counts[stage] = s_df.select(pls.col('session_id').n_unique()).item()
        cat_male_counts[stage] = s_df.filter(pls.col('sex') == 'male').height
        cat_female_counts[stage] = s_df.filter(pls.col('sex') == 'female').height
        for sess in s_df['session_id'].unique().to_list():
            sess_df = s_df.filter(pls.col('session_id') == sess)
            m_c = max(sess_df.filter(pls.col('sex') == 'male').height, 1)
            f_c = max(sess_df.filter(pls.col('sex') == 'female').height, 1)
            cat_ratios[stage].append(m_c / f_c)
    estrous_data[cat] = {
        'session_counts': cat_session_counts,
        'male_usv_counts': cat_male_counts,
        'female_usv_counts': cat_female_counts,
        'male_female_ratios': cat_ratios
    }

sorted_counts = {stage: session_counts.get(stage, 0) for stage in category_order}
print(f"Data aggregated. Total valid sessions mapped: {sum(session_counts.values())}")

# Plot 1: Pie chart
if sum(session_counts.values()) > 0:
    fig_pie, ax_pie, stats_pie = uss.plot_estrous_stage_pie_chart(
        session_counts=sorted_counts,
        label_map=label_map,
        slice_colors=estrous_colors
    )
    if save_fig_bool:
        fig_pie.savefig(figure_save_dir / f"{session_type}_estrous_stage_distribution_pie.{save_fig_format}", bbox_inches='tight')
    plt.show()

# Plot 2: USV rates bar
fig_rates, axes_rates, stats_rates = uss.plot_estrous_usv_rates(
    session_counts=session_counts,
    male_usv_counts=male_usv_counts,
    female_usv_counts=female_usv_counts,
    category_order=category_order,
    category_labels=category_labels,
    male_color=male_color,
    female_color=female_color,
    text_color='#202020'
)
if save_fig_bool:
    fig_rates.savefig(figure_save_dir / f"{session_type}_estrous_usv_rates_bar.{save_fig_format}", bbox_inches='tight')
plt.show()

# Plot 3: Ratio scatter
fig_ratio, ax_ratio, stats_ratio = uss.plot_estrous_ratio_scatter(
    ratio_dict=male_female_ratios,
    category_order=category_order,
    category_labels=category_labels,
    scatter_colors=estrous_colors,
    line_color='#202020',
    text_color='#000000',
    use_log_scale=True,
    confidence_level=0.99
)
if save_fig_bool:
    fig_ratio.savefig(figure_save_dir / f"{session_type}_estrous_usv_ratio_scatter.{save_fig_format}", bbox_inches='tight')
plt.show()

print("USV ratio statistical summary:")
for stage in category_order:
    if stage in stats_ratio:
        stage_stats = stats_ratio[stage]
        if stage_stats['n'] > 0:
            print(f"{label_map[stage]} (n={stage_stats['n']}): Mean = {stage_stats['mean']:.2f} \u00b1 {stage_stats['sem']:.2f}")


## 14. Estrous metrics broken down by category

Category-resolved version of the previous panel.


In [ ]:
### Plot category-specific estrous metrics

print("Generating estrous grid for USV rates...")

# Estrous rates facet grid
fig_estrous_rates, axes_estrous_rates, stats_estrous_rates = uss.plot_category_estrous_rates_grid(
    estrous_data=estrous_data,
    valid_stages=valid_stages,
    male_color=male_color,
    female_color=female_color
)

if save_fig_bool:
    rates_grid_path = figure_save_dir / f"{session_type}_usv_category_estrous_rates_grid.{save_fig_format}"
    fig_estrous_rates.savefig(rates_grid_path, bbox_inches='tight')
plt.show()

## Estrous ratio facet grids (category-wise and stage-wise)
print("Generating estrous grids for Male-to-Female ratios...")

# The function now returns (fig_cats, fig_stages), (axes_cats, axes_stages), and stats
(fig_ratio_cats, fig_ratio_stages), (axes_ratio_cats, axes_ratio_stages), stats_estrous_ratio = uss.plot_category_estrous_ratio_grid(
    estrous_data=estrous_data,
    valid_stages=valid_stages,
    scatter_colors=['#810000', '#ff1714', '#ff5555', '#ffaaaa']
)

# Save and show figure 1: breakdown by category
if save_fig_bool:
    cat_ratio_path = figure_save_dir / f"{session_type}_usv_ratio_by_category_grid.{save_fig_format}"
    fig_ratio_cats.savefig(cat_ratio_path, bbox_inches='tight')
print("Figure 1: Ratios broken down by USV Category")
plt.figure(fig_ratio_cats.number)
plt.show()

# Save and show figure 2: breakdown by estrous stage
if save_fig_bool:
    stage_ratio_path = figure_save_dir / f"{session_type}_usv_ratio_by_estrous_stage_grid.{save_fig_format}"
    fig_ratio_stages.savefig(stage_ratio_path, bbox_inches='tight')
print("Figure 2: Ratios broken down by Estrous Stage")
plt.figure(fig_ratio_stages.number)
plt.show()

## 15. Spatial vocalization distributions (polar KDEs)

Where in the arena, relative to each animal's body axis, do USVs tend to be emitted? Polar KDEs make rostral / caudal asymmetries easy to see.


In [ ]:
### Plot spatial vocalization distributions (polar KDEs)

max_plot_distance = 20.0  # Limit to 20cm for near-field social interaction
occupancy_thresh = 0.001  # Minimum background density required to plot normalized likelihood
max_points = 50000        # Caps background frames for KDE speed

print("Plotting male spatial vocalization distribution...")

male_usv = usv_pls.filter(pls.col('sex') == 'male').select(['distance', 'mf_angle']).drop_nulls()
female_usv = usv_pls.filter(pls.col('sex') == 'female').select(['distance', 'fm_angle']).drop_nulls()
all_dist = bg_pls['distance'].drop_nulls().to_numpy()
all_mf_angle = bg_pls['mf_angle'].drop_nulls().to_numpy()
all_fm_angle = bg_pls['fm_angle'].drop_nulls().to_numpy()

fig_polar_m, axes_polar_m, stats_polar_m = uss.plot_polar_kde_distance_angle(
    usv_distances=male_usv['distance'].to_numpy(),
    usv_angles_deg=male_usv['mf_angle'].to_numpy(),
    all_distances=all_dist,
    all_angles_deg=all_mf_angle,
    max_distance=max_plot_distance,
    colormap=female_cmap,
    ylabel="nose-nose distance (cm)",
    occupancy_threshold=occupancy_thresh,
    max_kde_points=max_points
)
if save_fig_bool:
    fig_polar_m.savefig(figure_save_dir / f"{session_type}_male_polar_kde_spatial_distribution.{save_fig_format}", bbox_inches='tight')
plt.show()

print("Plotting female spatial vocalization distribution...")

fig_polar_f, axes_polar_f, stats_polar_f = uss.plot_polar_kde_distance_angle(
    usv_distances=female_usv['distance'].to_numpy(),
    usv_angles_deg=female_usv['fm_angle'].to_numpy(),
    all_distances=all_dist,
    all_angles_deg=all_fm_angle,
    max_distance=max_plot_distance,
    colormap=male_cmap,
    ylabel="nose-nose distance (cm)",
    occupancy_threshold=occupancy_thresh,
    max_kde_points=max_points
)
if save_fig_bool:
    fig_polar_f.savefig(figure_save_dir / f"{session_type}_female_polar_kde_spatial_distribution.{save_fig_format}", bbox_inches='tight')
plt.show()


## 16. Spatial likelihood grid by category

Per-category 2D heatmaps of where in the arena USVs of each type were emitted.


In [ ]:
### Plot spatial likelihood grid for each USV category

usv_categories_list = sorted(usv_df['category'].dropna().unique().tolist())

# Reconstruct global_behavior_metrics from master DataFrames for plot_category_polar_kde_grid
global_behavior_metrics = {
    'all_frames': {
        'distance': bg_pls['distance'].drop_nulls().to_list(),
        'mf_angle': bg_pls['mf_angle'].drop_nulls().to_list(),
        'fm_angle': bg_pls['fm_angle'].drop_nulls().to_list()
    }
}
for cat in usv_categories_list:
    cat_df = usv_pls.filter(pls.col('category') == cat)
    male_df = cat_df.filter(pls.col('sex') == 'male').select(['distance', 'mf_angle', 'duration']).drop_nulls()
    female_df = cat_df.filter(pls.col('sex') == 'female').select(['distance', 'fm_angle', 'duration']).drop_nulls()
    unassigned_df = cat_df.filter(pls.col('sex') == 'unassigned').select(['distance', 'duration']).drop_nulls()
    global_behavior_metrics[cat] = {
        'male': {
            'distance': male_df['distance'].to_list(),
            'mf_angle': male_df['mf_angle'].to_list(),
            'usv_duration': male_df['duration'].to_list()
        },
        'female': {
            'distance': female_df['distance'].to_list(),
            'fm_angle': female_df['fm_angle'].to_list(),
            'usv_duration': female_df['duration'].to_list()
        },
        'unassigned': {
            'distance': unassigned_df['distance'].to_list(),
            'usv_duration': unassigned_df['duration'].to_list()
        }
    }

# Plot males
print("Generating Male polar KDE grid...")
fig_polar_m, axes_polar_m, stats_polar_m = uss.plot_category_polar_kde_grid(
    global_behavior_metrics=global_behavior_metrics,
    sex_key='male',
    max_distance=max_plot_distance,
    threshold=100,
    colormap=female_cmap
)
if save_fig_bool:
    fig_polar_m.savefig(figure_save_dir / f"{session_type}_male_category_likelihood_grid.{save_fig_format}", bbox_inches='tight')
plt.show()

# Plot females
print("Generating Female polar KDE grid...")
fig_polar_f, axes_polar_f, stats_polar_f = uss.plot_category_polar_kde_grid(
    global_behavior_metrics=global_behavior_metrics,
    sex_key='female',
    max_distance=max_plot_distance,
    threshold=50,
    colormap=male_cmap
)
if save_fig_bool:
    fig_polar_f.savefig(figure_save_dir / f"{session_type}_female_category_likelihood_grid.{save_fig_format}", bbox_inches='tight')
plt.show()


## 17. Spatial likelihood, category × estrous-stage

Final panel: spatial heatmaps faceted simultaneously by USV category and estrous stage. This is the highest-resolution joint summary in the notebook; expect sparse cells in rare combinations.


In [ ]:
### Plot USV category X estrous stage spatial likelihood grid

# Attach estrous stage if not already present (e.g. if cell 13 was not run)
if 'estrous_stage' not in usv_pls.columns:
    usv_pls = usv_pls.with_columns(
        pls.col('experiment_code').str.slice(-1).alias('estrous_stage')
    )

max_plot_distance = 20.0
occupancy_thresh = 0.01   # relative: 1% of background KDE peak
min_points = 30           # minimum USVs per (category, stage) cell to plot

valid_stages = ['p', 'e', 'm', 'd']
label_map = {
    'p': 'Proestrus',
    'e': 'Estrus',
    'm': 'Metestrus',
    'd': 'Diestrus'
}

# Male
print("Generating Male Category × Estrous Stage KDE grid...")
fig_ek_m, axes_ek_m, stats_ek_m = uss.plot_estrous_category_kde_grid(
    usv_pls=usv_pls,
    bg_pls=bg_pls,
    sex_key='male',
    valid_stages=valid_stages,
    stage_label_map=label_map,
    max_distance=max_plot_distance,
    occupancy_threshold=occupancy_thresh,
    colormap=female_cmap,
    threshold=min_points,
    max_kde_points=50000
)
if save_fig_bool:
    fig_ek_m.savefig(figure_save_dir / f"{session_type}_male_estrous_category_kde_grid.{save_fig_format}", bbox_inches='tight')
plt.show()

# Female
print("Generating Female Category × Estrous Stage KDE grid...")
fig_ek_f, axes_ek_f, stats_ek_f = uss.plot_estrous_category_kde_grid(
    usv_pls=usv_pls,
    bg_pls=bg_pls,
    sex_key='female',
    valid_stages=valid_stages,
    stage_label_map=label_map,
    max_distance=max_plot_distance,
    occupancy_threshold=occupancy_thresh,
    colormap=male_cmap,
    threshold=min_points,
    max_kde_points=50000
)
if save_fig_bool:
    fig_ek_f.savefig(figure_save_dir / f"{session_type}_female_estrous_category_kde_grid.{save_fig_format}", bbox_inches='tight')
plt.show()
